In [2]:
!pip install seaborn

'pip' is not recognized as an internal or external command,
operable program or batch file.


In [1]:
import seaborn as sns
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, r2_score
from tabulate import tabulate
import time

ModuleNotFoundError: No module named 'seaborn'

In [26]:

df = sns.load_dataset("titanic")

# Hyperparameters

In [27]:
TARGET_COL = "fare"
BATCH_SIZE = 64
EPOCHS = 50
ALPHA = 1e-3  

# Drop NaN in target and Inf

In [28]:
df = df.dropna(subset=[TARGET_COL])
df = df.replace([np.inf, -np.inf], np.nan)
df = df.dropna(subset=[TARGET_COL])

# Encoding

In [29]:
df_encoded = pd.get_dummies(df, drop_first=True)

In [30]:
X = df_encoded.drop(columns=[TARGET_COL]).values
y = df_encoded[TARGET_COL].values

In [31]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)


# Scaling

In [32]:
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test  = scaler.transform(X_test)

# NaN / Inf tozalash

In [33]:
X_train = np.nan_to_num(X_train, nan=0.0, posinf=0.0, neginf=0.0)
X_test  = np.nan_to_num(X_test,  nan=0.0, posinf=0.0, neginf=0.0)
y_train = np.nan_to_num(y_train, nan=0.0, posinf=0.0, neginf=0.0)
y_test  = np.nan_to_num(y_test,  nan=0.0, posinf=0.0, neginf=0.0)

# Log transform target

In [34]:
y_train_log = np.log1p(y_train)
y_test_log  = np.log1p(y_test)

# Convert to PyTorch tensors

In [35]:
X_train_t = torch.tensor(X_train, dtype=torch.float32)
y_train_t = torch.tensor(y_train_log, dtype=torch.float32).view(-1, 1)
X_test_t  = torch.tensor(X_test, dtype=torch.float32)
y_test_t  = torch.tensor(y_test_log, dtype=torch.float32).view(-1, 1)

In [36]:
train_ds = TensorDataset(X_train_t, y_train_t)
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True)

# Neural Network model

In [37]:
torch.manual_seed(0)
input_dim = X_train_t.shape[1]

model = nn.Sequential(
    nn.Linear(input_dim, 64),
    nn.ReLU(),
    nn.Linear(64, 1)
)

loss_fn = nn.MSELoss()
opt = optim.SGD(model.parameters(), lr=ALPHA)

# Train Neural Network

In [38]:

for epoch in range(1, EPOCHS + 1):
    model.train()
    total_loss = 0.0
    for bx, by in train_loader:
        pred = model(bx)
        loss = loss_fn(pred, by)
        opt.zero_grad()
        loss.backward()
        opt.step()
        total_loss += loss.item()
    if epoch % 10 == 0:
        print(f"Epoch {epoch:3d} | Train Loss: {total_loss/len(train_loader):.4f}")

Epoch  10 | Train Loss: 1.4844
Epoch  20 | Train Loss: 0.6826
Epoch  30 | Train Loss: 0.5433
Epoch  40 | Train Loss: 0.4896
Epoch  50 | Train Loss: 0.4356


# Predict with Neural Network (SDL)

In [ ]:
model.eval()
with torch.no_grad():
    pred_nn_log = model(X_test_t).cpu().numpy()
    pred_nn = np.expm1(pred_nn_log)  
mae_nn = mean_absolute_error(y_test, pred_nn)
r2_nn  = r2_score(y_test, pred_nn)

print(f"SDL: NeuralNet | MAE: {mae_nn:.2f} | R2: {r2_nn:.4f}")

SDL: NeuralNet | MAE: 15.89 | R2: 0.4224


# Classical ML (SML)

In [40]:
def regression_report(name, y_true, y_pred, train_time):
    mae = mean_absolute_error(y_true, y_pred)
    r2 = r2_score(y_true, y_pred)
    print(f"{name:18s} | MAE: {mae:.2f} | R2: {r2:.4f} | time: {train_time:.3f}s")

In [44]:
t0 = time.time()
lr = LinearRegression()
lr.fit(X_train, y_train)
pred_lr = lr.predict(X_test)
regression_report("SML: LinearReg", y_test, pred_lr, time.time()-t0)
t0 = time.time()
ridge = Ridge(alpha=1.0)
ridge.fit(X_train, y_train)
pred_ridge = ridge.predict(X_test)
regression_report("SML: Ridge", y_test, pred_ridge, time.time()-t0)
t0 = time.time()
rf = RandomForestRegressor(n_estimators=300, random_state=42)
rf.fit(X_train, y_train)
pred_rf = rf.predict(X_test)
regression_report("SML: RandomForest", y_test, pred_rf, time.time()-t0)

SML: LinearReg     | MAE: 17.05 | R2: 0.4932 | time: 0.004s
SML: Ridge         | MAE: 17.02 | R2: 0.4945 | time: 0.003s
SML: RandomForest  | MAE: 14.02 | R2: 0.2487 | time: 0.709s


In [45]:
results = [
    ["Linear Regression",        "SML", mean_absolute_error(y_test, pred_lr), r2_score(y_test, pred_lr)],
    ["Ridge Regression",         "SML", mean_absolute_error(y_test, pred_ridge), r2_score(y_test, pred_ridge)],
    ["Random Forest",            "SML", mean_absolute_error(y_test, pred_rf), r2_score(y_test, pred_rf)],
    ["Neural Network (PyTorch)", "SDL", mae_nn, r2_nn],
]

headers = ["Model", "Type", "MAE", "R2"]
print(tabulate(results, headers=headers, tablefmt="grid", floatfmt=".4f"))

+--------------------------+--------+---------+--------+
| Model                    | Type   |     MAE |     R2 |
+==========================+========+=========+========+
| Linear Regression        | SML    | 17.0478 | 0.4932 |
+--------------------------+--------+---------+--------+
| Ridge Regression         | SML    | 17.0175 | 0.4945 |
+--------------------------+--------+---------+--------+
| Random Forest            | SML    | 14.0197 | 0.2487 |
+--------------------------+--------+---------+--------+
| Neural Network (PyTorch) | SDL    | 15.8874 | 0.4224 |
+--------------------------+--------+---------+--------+


## Xulosa

1. **Linear va Ridge Regression**  
   - Eng oddiy modellardir.  
   - R² ~ 0.49 bilan datasetdagi o‘zgarishlarning yarmini tushuntiradi.  
   - MAE ~ 17, ya’ni o‘rtacha xato qiymati $17 atrofida.  

2. **Random Forest**  
   - MAE eng past (14.02), ya’ni xato qiymati kamaygan.  
   - Ammo R² juda past (0.25), bu model ba’zi targetlarni yaxshi bashorat qilsa-da, umumiy dispersiyani tushuntirmaydi.  

3. **Neural Network (PyTorch, SDL)**  
   - MAE va R² o‘rtacha, R² = 0.42.  
   - Model murakkabroq bo‘lsa ham, Titanic dataset kabi kichik va noaniq regression muammosida SML modellar bilan katta farq qilmaydi.  

### 🔹 Umumiy xulosa

- **Titanic dataset** regression uchun kichik va cheklangan bo‘lgani sababli, **oddiy SML modellari (Linear/Ridge)** ko‘pincha Neural Network dan yaxshi yoki shunchalik samarali ishlaydi.  
- Neural Network faqatgina katta va boy feature setlarda, yoki no-linear munosabatlarda ustunlik beradi.  
- MAE va R² birliklariga qarab, **Linear/Ridge** modellari bu dataset uchun yetarli va barqaror yechim.